# FASE 4: Pemodelan Deep Learning (LSTM & BiLSTM)
## Prediksi PM2.5 di Cekungan Bandung
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini melatih dua arsitektur Deep Learning berbasis *Recurrent Neural Network*:
1. **Stacked LSTM** — membaca urutan waktu dari satu arah (maju)
2. **BiLSTM (Bidirectional LSTM)** — membaca urutan waktu dari dua arah (maju dan mundur)

Kedua model menggunakan **data Validasi eksplisit** untuk *Early Stopping* dan pemilihan arsitektur terbaik.

**Target:** Mengalahkan skor RMSE baseline terbaik (RF Tuned / XGB Tuned) dari Notebook 03.


## 1. Import Library


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import time

# Evaluasi
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TensorFlow / Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print("[OK] Library loaded.")


## 2. Load Data Processed & Scaler


In [ ]:
DATA_DIR = '../data/processed'

train_scaled = pd.read_csv(os.path.join(DATA_DIR, 'train_scaled.csv'), index_col=0, parse_dates=True)
val_scaled   = pd.read_csv(os.path.join(DATA_DIR, 'val_scaled.csv'), index_col=0, parse_dates=True)
test_scaled  = pd.read_csv(os.path.join(DATA_DIR, 'test_scaled.csv'), index_col=0, parse_dates=True)

scaler_y = joblib.load(os.path.join(DATA_DIR, 'scaler_y.pkl'))

target_col = 'pm25'
print(f"Train : {train_scaled.shape}")
print(f"Val   : {val_scaled.shape}")
print(f"Test  : {test_scaled.shape}")


## 3. Transformasi Data: Tabel 2D menjadi Tensor 3D (*Sliding Window*)

LSTM membutuhkan input berbentuk **Tensor 3D**: `[Jumlah Sampel, Jendela Waktu, Jumlah Fitur]`.

Parameter `TIME_STEPS = 24` berarti untuk memprediksi PM2.5 di jam ke-25, model diberi konteks cuaca dan polusi selama **24 jam sebelumnya** (1 hari penuh).


In [ ]:
def create_sequences(df, target_col, time_steps=24):
    """Mengubah DataFrame 2D menjadi Tensor 3D untuk LSTM."""
    X, y = [], []
    data_array = df.values
    target_idx = df.columns.get_loc(target_col)

    for i in range(len(data_array) - time_steps):
        X.append(data_array[i:(i + time_steps)])
        y.append(data_array[i + time_steps, target_idx])

    return np.array(X), np.array(y)

TIME_STEPS = 24

X_train_seq, y_train_seq = create_sequences(train_scaled, target_col, TIME_STEPS)
X_val_seq,   y_val_seq   = create_sequences(val_scaled, target_col, TIME_STEPS)
X_test_seq,  y_test_seq  = create_sequences(test_scaled, target_col, TIME_STEPS)

n_features = X_train_seq.shape[2]

print(f"Tensor X_train : {X_train_seq.shape} --> [Sampel, TimeSteps, Fitur]")
print(f"Tensor X_val   : {X_val_seq.shape}")
print(f"Tensor X_test  : {X_test_seq.shape}")
print(f"Jumlah fitur   : {n_features}")


## 4. Model 1: Stacked LSTM

### Arsitektur:
```
Input         : (24, n_fitur)
LSTM Layer 1  : 64 unit, return_sequences=True
Dropout       : 0.2
LSTM Layer 2  : 32 unit, return_sequences=False
Dropout       : 0.2
Dense Output  : 1 unit
```

Data **Validasi eksplisit** digunakan untuk `EarlyStopping` (bukan `validation_split` dari data Training).


In [ ]:
def build_lstm(n_timesteps, n_features):
    """Membangun arsitektur Stacked LSTM."""
    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))
    model.add(LSTM(units=64, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(units=32, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

lstm_model = build_lstm(TIME_STEPS, n_features)
lstm_model.summary()


In [ ]:
print("Melatih Stacked LSTM...")

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

start = time.time()
lstm_history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_seq, y_val_seq),  # Validasi eksplisit
    callbacks=[early_stop],
    verbose=1
)
lstm_train_time = time.time() - start
print(f"\nTraining LSTM selesai ({lstm_train_time:.1f} detik)")


## 5. Model 2: BiLSTM (Bidirectional LSTM)

BiLSTM membaca urutan waktu dari **dua arah sekaligus** (maju dan mundur). Dalam konteks prediksi PM2.5, ini memungkinkan model untuk menangkap hubungan temporal yang lebih kaya — misalnya, kondisi cuaca 24 jam ke belakang mungkin memiliki pola yang saling bergantung di kedua ujung jendela waktu.

### Arsitektur:
```
Input              : (24, n_fitur)
Bidirectional LSTM : 64 unit (128 total output), return_sequences=True
Dropout            : 0.2
Bidirectional LSTM : 32 unit (64 total output), return_sequences=False
Dropout            : 0.2
Dense Output       : 1 unit
```


In [ ]:
def build_bilstm(n_timesteps, n_features):
    """Membangun arsitektur Bidirectional LSTM."""
    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))
    model.add(Bidirectional(LSTM(units=64, return_sequences=True)))
    model.add(Dropout(0.2))
    model.add(Bidirectional(LSTM(units=32, return_sequences=False)))
    model.add(Dropout(0.2))
    model.add(Dense(units=1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

bilstm_model = build_bilstm(TIME_STEPS, n_features)
bilstm_model.summary()


In [ ]:
print("Melatih Bidirectional LSTM...")

early_stop_bi = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

start = time.time()
bilstm_history = bilstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_seq, y_val_seq),
    callbacks=[early_stop_bi],
    verbose=1
)
bilstm_train_time = time.time() - start
print(f"\nTraining BiLSTM selesai ({bilstm_train_time:.1f} detik)")


## 7. Evaluasi Akhir pada Data Testing


In [ ]:
def evaluate_dl_model(model_name, model, X_seq, y_seq):
    """Mengevaluasi model DL: prediksi, inversi, hitung metrik."""
    y_pred_s = model.predict(X_seq, verbose=0)
    y_true_inv = scaler_y.inverse_transform(y_seq.reshape(-1, 1)).flatten()
    y_pred_inv = scaler_y.inverse_transform(y_pred_s).flatten()

    rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
    mae  = mean_absolute_error(y_true_inv, y_pred_inv)
    r2   = r2_score(y_true_inv, y_pred_inv)

    print(f"=== Kinerja Model: {model_name} ===")
    print(f"  RMSE : {rmse:.4f} ug/m3")
    print(f"  MAE  : {mae:.4f} ug/m3")
    print(f"  R2   : {r2:.4f}")

    return {'name': model_name, 'rmse': rmse, 'mae': mae, 'r2': r2,
            'y_true': y_true_inv, 'y_pred': y_pred_inv}

print("--- Evaluasi pada Data Testing ---\n")

lstm_result   = evaluate_dl_model("Stacked LSTM", lstm_model, X_test_seq, y_test_seq)
print()
bilstm_result = evaluate_dl_model("BiLSTM", bilstm_model, X_test_seq, y_test_seq)

# Tentukan pemenang
if lstm_result['rmse'] <= bilstm_result['rmse']:
    winner_name = "Stacked LSTM"
    winner_model = lstm_model
    winner_result = lstm_result
else:
    winner_name = "BiLSTM"
    winner_model = bilstm_model
    winner_result = bilstm_result

print(f"\n>> Model Deep Learning Terbaik: {winner_name} (RMSE = {winner_result['rmse']:.4f})")


## 4A. Hyperparameter Tuning (Grid Search Manual)

Bagian 4 dan 5 di atas melatih model LSTM dan BiLSTM dengan arsitektur **default** (64-32 neuron, dropout 0.2, batch 32) dan hanya mengandalkan *Early Stopping* untuk mengatur jumlah epoch.

Agar perbandingan dengan Baseline Models (RF & XGBoost yang sudah di-tuning via GridSearch) menjadi **adil (*fair comparison*)**, kita perlu melakukan pencarian parameter terbaik untuk Deep Learning juga.

### Parameter yang Di-Tuning:
| Parameter | Opsi yang Dicoba | Alasan |
|-----------|-----------------|--------|
| `units` (jumlah neuron) | [32, 64, 128] | Mengatur kapasitas model menangkap pola kompleks |
| `dropout` | [0.1, 0.2, 0.3] | Mengatur tingkat regularisasi untuk mencegah *overfitting* |
| `batch_size` | [16, 32, 64] | Mengatur kestabilan proses belajar (*gradient update*) |

Total kombinasi: **3 × 3 × 3 = 27 konfigurasi per arsitektur**.

Setiap konfigurasi tetap menggunakan `EarlyStopping` pada data Validasi. Model terbaik dipilih berdasarkan **RMSE terendah pada Validation Set**.


In [ ]:
# ============================================================
# GRID SEARCH MANUAL — Stacked LSTM
# ============================================================

from itertools import product
import tensorflow as tf

# Definisikan ruang pencarian hyperparameter
param_grid = {
    'units': [32, 64, 128],
    'dropout': [0.1, 0.2, 0.3],
    'batch_size': [16, 32, 64]
}

# Buat semua kombinasi
combinations = list(product(param_grid['units'], param_grid['dropout'], param_grid['batch_size']))
print(f"Total kombinasi LSTM yang akan diuji: {len(combinations)}")

# Variabel untuk menyimpan hasil terbaik
best_lstm_val_rmse = float('inf')
best_lstm_params = {}
best_lstm_tuned_model = None
tuning_results_lstm = []

for idx, (units, dropout, batch_size) in enumerate(combinations):
    print(f"\n[{idx+1}/{len(combinations)}] units={units}, dropout={dropout}, batch={batch_size}", end=' ')

    # Bangun model baru setiap iterasi
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(Input(shape=(TIME_STEPS, n_features)))
    model.add(LSTM(units=units, return_sequences=True))
    model.add(Dropout(dropout))
    model.add(LSTM(units=units // 2, return_sequences=False))
    model.add(Dropout(dropout))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')

    # Latih dengan Early Stopping
    es = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=0)
    model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=batch_size,
        validation_data=(X_val_seq, y_val_seq),
        callbacks=[es],
        verbose=0
    )

    # Evaluasi di Validation Set (BUKAN Test Set)
    y_pred_val = model.predict(X_val_seq, verbose=0)
    y_val_inv  = scaler_y.inverse_transform(y_val_seq.reshape(-1, 1)).flatten()
    y_pred_inv = scaler_y.inverse_transform(y_pred_val).flatten()
    val_rmse = np.sqrt(mean_squared_error(y_val_inv, y_pred_inv))

    print(f"-> Val RMSE: {val_rmse:.4f}")
    tuning_results_lstm.append({'units': units, 'dropout': dropout, 'batch_size': batch_size, 'val_rmse': val_rmse})

    if val_rmse < best_lstm_val_rmse:
        best_lstm_val_rmse = val_rmse
        best_lstm_params = {'units': units, 'dropout': dropout, 'batch_size': batch_size}
        best_lstm_tuned_model = model

print(f"\n{'='*60}")
print(f"LSTM Tuned — Parameter Terbaik: {best_lstm_params}")
print(f"LSTM Tuned — Val RMSE Terbaik : {best_lstm_val_rmse:.4f}")
print(f"{'='*60}")


In [ ]:
# ============================================================
# GRID SEARCH MANUAL — BiLSTM
# ============================================================

best_bilstm_val_rmse = float('inf')
best_bilstm_params = {}
best_bilstm_tuned_model = None
tuning_results_bilstm = []

for idx, (units, dropout, batch_size) in enumerate(combinations):
    print(f"\n[{idx+1}/{len(combinations)}] units={units}, dropout={dropout}, batch={batch_size}", end=' ')

    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(Input(shape=(TIME_STEPS, n_features)))
    model.add(Bidirectional(LSTM(units=units, return_sequences=True)))
    model.add(Dropout(dropout))
    model.add(Bidirectional(LSTM(units=units // 2, return_sequences=False)))
    model.add(Dropout(dropout))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')

    es = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=0)
    model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=batch_size,
        validation_data=(X_val_seq, y_val_seq),
        callbacks=[es],
        verbose=0
    )

    y_pred_val = model.predict(X_val_seq, verbose=0)
    y_val_inv  = scaler_y.inverse_transform(y_val_seq.reshape(-1, 1)).flatten()
    y_pred_inv = scaler_y.inverse_transform(y_pred_val).flatten()
    val_rmse = np.sqrt(mean_squared_error(y_val_inv, y_pred_inv))

    print(f"-> Val RMSE: {val_rmse:.4f}")
    tuning_results_bilstm.append({'units': units, 'dropout': dropout, 'batch_size': batch_size, 'val_rmse': val_rmse})

    if val_rmse < best_bilstm_val_rmse:
        best_bilstm_val_rmse = val_rmse
        best_bilstm_params = {'units': units, 'dropout': dropout, 'batch_size': batch_size}
        best_bilstm_tuned_model = model

print(f"\n{'='*60}")
print(f"BiLSTM Tuned — Parameter Terbaik: {best_bilstm_params}")
print(f"BiLSTM Tuned — Val RMSE Terbaik : {best_bilstm_val_rmse:.4f}")
print(f"{'='*60}")


In [ ]:
# ============================================================
# RANGKUMAN HASIL TUNING: Tabel Perbandingan
# ============================================================

df_lstm_tuning = pd.DataFrame(tuning_results_lstm).sort_values('val_rmse')
df_bilstm_tuning = pd.DataFrame(tuning_results_bilstm).sort_values('val_rmse')

print("\n" + "="*60)
print("TOP-5 KONFIGURASI TERBAIK — Stacked LSTM")
print("="*60)
print(df_lstm_tuning.head().to_string(index=False))

print("\n" + "="*60)
print("TOP-5 KONFIGURASI TERBAIK — BiLSTM")
print("="*60)
print(df_bilstm_tuning.head().to_string(index=False))

# Visualisasi heatmap tuning (units vs dropout, rata-rata batch)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_t, title in zip(axes, [df_lstm_tuning, df_bilstm_tuning], ['LSTM', 'BiLSTM']):
    pivot = df_t.groupby(['units', 'dropout'])['val_rmse'].mean().unstack()
    im = ax.imshow(pivot.values, cmap='YlOrRd_r', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{d}' for d in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{u}' for u in pivot.index])
    ax.set_xlabel('Dropout')
    ax.set_ylabel('Units')
    ax.set_title(f'{title} — Val RMSE (rata-rata batch)', fontweight='bold')
    # Anotasi angka
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f'{pivot.values[i, j]:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()


### 4B. Perbandingan: Default (Early Stopping Only) vs Tuned (Grid Search)

Tabel di bawah ini akan membandingkan performa model **Default** (Bagian 4 & 5) dengan model **Tuned** (Bagian 4A) pada **Test Set**.

Ini menunjukkan apakah *hyperparameter tuning* memberikan peningkatan signifikan dibanding arsitektur yang dipilih secara heuristik.


In [ ]:
# ============================================================
# EVALUASI AKHIR: Default vs Tuned pada DATA TESTING
# ============================================================

print("="*65)
print("PERBANDINGAN DEFAULT vs TUNED — Evaluasi pada Data Testing")
print("="*65)

# Model Default sudah dievaluasi di Bagian 7 (lstm_result & bilstm_result)
# Sekarang evaluasi model Tuned
lstm_tuned_result   = evaluate_dl_model("LSTM Tuned", best_lstm_tuned_model, X_test_seq, y_test_seq)
print()
bilstm_tuned_result = evaluate_dl_model("BiLSTM Tuned", best_bilstm_tuned_model, X_test_seq, y_test_seq)

# Buat tabel rangkuman
comparison_data = {
    'Model': ['LSTM Default', 'LSTM Tuned', 'BiLSTM Default', 'BiLSTM Tuned'],
    'RMSE': [
        lstm_result['rmse'], lstm_tuned_result['rmse'],
        bilstm_result['rmse'], bilstm_tuned_result['rmse']
    ],
    'MAE': [
        lstm_result['mae'], lstm_tuned_result['mae'],
        bilstm_result['mae'], bilstm_tuned_result['mae']
    ],
    'R2': [
        lstm_result['r2'], lstm_tuned_result['r2'],
        bilstm_result['r2'], bilstm_tuned_result['r2']
    ],
    'Params': [
        'units=64, drop=0.2, batch=32',
        f"units={best_lstm_params['units']}, drop={best_lstm_params['dropout']}, batch={best_lstm_params['batch_size']}",
        'units=64, drop=0.2, batch=32',
        f"units={best_bilstm_params['units']}, drop={best_bilstm_params['dropout']}, batch={best_bilstm_params['batch_size']}"
    ]
}

df_compare = pd.DataFrame(comparison_data)
print(f"\n{'='*65}")
print("TABEL RANGKUMAN PERBANDINGAN (Test Set)")
print(f"{'='*65}")
print(df_compare.to_string(index=False))

# Tentukan pemenang akhir dari semua 4 model
all_results = [
    ('LSTM Default', lstm_result, lstm_model),
    ('LSTM Tuned', lstm_tuned_result, best_lstm_tuned_model),
    ('BiLSTM Default', bilstm_result, bilstm_model),
    ('BiLSTM Tuned', bilstm_tuned_result, best_bilstm_tuned_model)
]

final_winner = min(all_results, key=lambda x: x[1]['rmse'])
winner_name = final_winner[0]
winner_result = final_winner[1]
winner_model = final_winner[2]

print(f"\n>> PEMENANG AKHIR: {winner_name} (RMSE = {winner_result['rmse']:.4f})")


## 6. Kurva Pembelajaran (*Learning Curves*)

Membandingkan proses belajar LSTM vs BiLSTM. Idealnya, *training loss* dan *validation loss* harus turun bersamaan tanpa divergensi besar (tanda *overfitting*).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# LSTM
axes[0].plot(lstm_history.history['loss'], label='Train Loss', color='royalblue')
axes[0].plot(lstm_history.history['val_loss'], label='Val Loss', color='darkorange')
axes[0].set_title('Stacked LSTM - Learning Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# BiLSTM
axes[1].plot(bilstm_history.history['loss'], label='Train Loss', color='royalblue')
axes[1].plot(bilstm_history.history['val_loss'], label='Val Loss', color='darkorange')
axes[1].set_title('BiLSTM - Learning Curve', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Visualisasi Prediksi vs Aktual


In [ ]:
# Ambil 14 hari pertama
subset_len = min(336, len(y_test_seq))
time_axis  = test_scaled.index[TIME_STEPS : TIME_STEPS + subset_len]

actual     = lstm_result['y_true'][:subset_len]
lstm_pred  = lstm_result['y_pred'][:subset_len]
bi_pred    = bilstm_result['y_pred'][:subset_len]

plt.figure(figsize=(16, 6))
plt.plot(time_axis, actual,    label='Aktual (PM2.5)', color='black', linewidth=2)
plt.plot(time_axis, lstm_pred, label='Stacked LSTM', color='magenta', alpha=0.8, linestyle='--')
plt.plot(time_axis, bi_pred,   label='BiLSTM', color='teal', alpha=0.8, linestyle='-.')

plt.title("LSTM vs BiLSTM vs Aktual (2 Minggu Pertama Data Test)", fontweight='bold')
plt.ylabel("PM2.5 (ug/m3)")
plt.xlabel("Waktu")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Menyimpan Model Pemenang

Model terbaik akan disimpan dan digunakan oleh **Notebook 05 (SHAP Interpretation)** untuk membongkar kontribusi setiap fitur.


In [ ]:
MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Simpan kedua model
lstm_model.save(os.path.join(MODEL_DIR, 'lstm_model.keras'))
bilstm_model.save(os.path.join(MODEL_DIR, 'bilstm_model.keras'))

# Simpan model pemenang dengan nama khusus untuk kemudahan di Notebook 05
winner_model.save(os.path.join(MODEL_DIR, 'best_dl_model.keras'))

# Simpan metadata pemenang
winner_info = {
    'model_name': winner_name,
    'rmse': winner_result['rmse'],
    'mae': winner_result['mae'],
    'r2': winner_result['r2']
}
joblib.dump(winner_info, os.path.join(MODEL_DIR, 'best_dl_info.pkl'))

print(f"[OK] Semua model berhasil disimpan di: {MODEL_DIR}")
print(f"  -> lstm_model.keras")
print(f"  -> bilstm_model.keras")
print(f"  -> best_dl_model.keras  ({winner_name})")
print(f"  -> best_dl_info.pkl")
